# 공공 자전거 장기 연체 예측
- 목표 : 대여 시점 정보로 100분 초과 장기 연체(`is_overdue_100m`) 예측
- 핵심 질문 : "정확도 84%인데 왜 연체를 하나도 못 잡았나?"

## [STEP 1] 파생변수 & 정답지(Target)
- `duration_min` = `return_time - rent_time` (분 단위)
- `is_overdue_100m` = `duration_min > 100` 이면 1 (예측 대상)
- `hour`, `weekday` = `rent_time` 기준 시간대, 요일 → 연체 비율, "전부 정상" 예측 시 정확도 확인

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/clean_rentals.csv",parse_dates=["rent_time","return_time"])

print(df.shape)
print(df.dtypes)

(13501, 16)
rental_id                      str
bike_id                        str
user_id                        str
rent_time           datetime64[us]
return_time         datetime64[us]
distance_km                float64
fee                          int64
payment_method                 str
station_id                     str
bike_type                      str
gear_count                 float64
manufacture_year           float64
daily_fee                  float64
station_name                   str
district                       str
duration_min               float64
dtype: object


In [2]:
# 1. 대여시간(분)
diff = df["return_time"] - df["rent_time"]
df["duration_min"] = diff.dt.total_seconds() / 60

print("[대여 시간(분) 계산 결과 - 상위 5행]")
print(df[["rent_time", "return_time", "duration_min"]].head())

[대여 시간(분) 계산 결과 - 상위 5행]
            rent_time         return_time  duration_min
0 2022-04-18 20:28:00 2022-04-18 20:44:00          16.0
1 2022-05-19 09:53:00 2022-05-19 11:08:00          75.0
2 2022-09-07 16:20:00 2022-09-07 18:04:00         104.0
3 2022-10-11 06:39:00 2022-10-11 07:18:00          39.0
4 2022-07-11 10:15:00 2022-07-11 10:53:00          38.0


In [3]:
# 2. 정답지 : 100분 초과면 1
df["is_overdue_100m"] = (df["duration_min"] > 100).astype(int)

print("[연체 여부별 건수] 0 = 정상, 1 = 연체")
print(df["is_overdue_100m"].value_counts())

[연체 여부별 건수] 0 = 정상, 1 = 연체
is_overdue_100m
0    11136
1     2365
Name: count, dtype: int64


In [4]:
# 3. 대여 시점 기준 시간대, 요일
df["hour"] = df["rent_time"].dt.hour
df["weekday"] = df["rent_time"].dt.weekday

print("[시간대, 요일 추출 결과 - 상위 5행]")
print(df[["rent_time","hour","weekday"]].head())

[시간대, 요일 추출 결과 - 상위 5행]
            rent_time  hour  weekday
0 2022-04-18 20:28:00    20        0
1 2022-05-19 09:53:00     9        3
2 2022-09-07 16:20:00    16        2
3 2022-10-11 06:39:00     6        1
4 2022-07-11 10:15:00    10        0


In [5]:
# 4. 연체 비율, "전부 정상" 예측 시 정확도
overdue_ratio = df["is_overdue_100m"].mean()
all_normal_acc = 1 - overdue_ratio

print(f"전체 행 수 : {len(df):,}")
print(f"연체 비율 : {overdue_ratio:.2%}")
print(f"'전부 정상' 예측 시 정확도 : {all_normal_acc:.2%}")

전체 행 수 : 13,501
연체 비율 : 17.52%
'전부 정상' 예측 시 정확도 : 82.48%


## [STEP 2] users 결합 & 가입 기간(파생변수)
- `users.csv` 를 `user_id` 기준으로 결합 (`how`, `validate` 명시) → 결합 전후 행 수 비교
- 가입 기간(일) = `rent_time - join_date`

In [6]:
# 5. users 불러오기
users = pd.read_csv("data/users.csv", parse_dates=["join_date"])

# 6. 결합
before = len(df)

df = df.merge(
    users,
    on="user_id",
    how="left",
    validate="many_to_one"
)

print(f"결합 전: {before:,}행")
print(f"결합 후: {len(df):,}행")

결합 전: 13,501행
결합 후: 13,501행


In [7]:
# 7. 가입 기간(일) = 대여시점 - 가입일
df["member_days"] = (df["rent_time"] - df["join_date"]).dt.days

print("[가입 기간(일) 요약]")
print(df["member_days"].describe())

[가입 기간(일) 요약]
count    13501.000000
mean        49.932301
std        312.660220
min       -622.000000
25%       -207.000000
50%         42.000000
75%        312.000000
max        720.000000
Name: member_days, dtype: float64
